# Lab: Subset Selection Methods
## CMSE 381 - Spring 2022
## Feb 25, 2022



In this module we are going to test out the subset selections methods we discussed in class from Chapter 6.1.

In [ ]:
# Everyone's favorite standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
import statsmodels.api as sm
import time

# Loading in the data

Ok, here we go, let's play with a baseball data set. 

In [ ]:
hitters_df = pd.read_csv('Hitters.csv')
hitters_df.head()

Annoyingly enough we have some missing values in the data.

In [ ]:
print("Number of null values:", hitters_df["Salary"].isnull().sum())


So let's go clean those up

In [ ]:
# Print the dimensions of the original Hitters data (322 rows x 20 columns)
print("Dimensions of original data:", hitters_df.shape)

# Drop any rows the contain missing values, along with the player names
hitters_df = hitters_df.dropna().drop('Player', axis=1)

# Print the dimensions of the modified Hitters data (263 rows x 20 columns)
print("Dimensions of modified data:", hitters_df.shape)

# One last check: should return 0
print("Number of null values:", hitters_df["Salary"].isnull().sum())

# Best subset selection

Whew, ok, the data clean up is out of the way (you're welcome) so lets get to the fun part. First, let's play around with the `itertools` package which lets us automatically get subsets of things using the `combinations` function. 

In [ ]:
import itertools 

In [ ]:
my_set = ['apples','oranges','bananas','pomegranates','grapes']
for combo in itertools.combinations(my_set,3):
    print(combo)

Note that the code above produces an `iterator`, which means I really need to have it used inside of a for loop. If I just try to print it out, I get this:

In [ ]:
print(itertools.combinations(my_set,3))

&#9989; **<font color=red>Do this:</font>** Write a script that prints all subsets of `my_set` of size 1 through 4. 


In [ ]:
# Your code here #


## Back to the data
Ok, back to the data set.  We're going work with linear regression models using the available inputs to predict the player salaries. Our goal is to check all the subsets (up to a limit cuz the computation time is narsty) and find the best one, where `best` is defined as the lowest RSS

We're first going to set this bit up so that we have dummy variables for the three categorical variables. Turns out pandas can do all my work for me.

In [ ]:
dummies = pd.get_dummies(hitters_df[['League', 'Division', 'NewLeague']])
dummies

&#9989; **<font color=red>Q:</font>** 
- What do the column names in the `dummies` data frame mean?  
- What does a row with all 0 entries mean about the original data point?


In [ ]:
# Your code here, you might need to test some things out to figure out the answers above #


&#9989; **<font color=red>Do this:</font>** 
- Make a new data frame `y` with just the `Salary` data. 
- Make a new data frame `X` by dropping the original categorical variables from `hitters_df` and putting in the dummy variables for `League_N`, `Division_W`, and `NewLeague_N`.
    - Important: Why am I not using ALL the variables from the `dummies` data frame?


In [ ]:
# Your code here#


The function `processSubset` takes in a list of features, then trains the model using the portion of the `X` data frame. 

In [ ]:
def processSubset(feature_set):
    # Fit model on feature_set and calculate RSS
    model = sm.OLS(y,X[list(feature_set)])
    regr = model.fit()
    RSS = ((regr.predict(X[list(feature_set)]) - y) ** 2).sum()
    return {"model":regr, "RSS":RSS}

&#9989; **<font color=red>Do this:</font>** Use the `processSubset` function to get the model and RSS for the model where we predict salary using `RBI`, `CWalks`, and `League_N`. 


In [ ]:
# Your code here #

This next function, `getBest` should take an integer `k` and create a data frame with all the resulting models and RSS values for each possible subset of size `k`. In the notation from the last class, this functing is finding $\mathcal{M}_k$.

&#9989; **<font color=red>Do this:</font>** Fix the code below to actually work. Which two variables are used by the best model $\mathcal{M}_2$?


In [ ]:
#---- you modify this function to actually work -----#
def getBest(k):
    
    # This function just keeps track of time for me 
    tic = time.time()
    
    results = []
    
    #---------Your code here--------#
    # use the itertools stuff from above to get all subsets
    # append the results of processSubset for each set 
    # to the list results
    #
    # Delete the line below here, it's just a placeholder so that the code 
    # runs before you add your code 
    results.append({'model':None,"RSS":42}) # DELETE ME!!! #
    #-------------------------------#
    
    
    # Wrap everything up in a nice dataframe
    models = pd.DataFrame(results)
    
    # Choose the model with the highest RSS 
    best_model = models.loc[models['RSS'].argmin()]
    
    toc = time.time()
    print("Processed", models.shape[0], "models on", k, "predictors in", (toc-tic), "seconds.")
    
    # Return the best model, along with some other useful information about the model
    return best_model


# Tester down here
getBest(2)

Now that you've got that working, let's find $\mathcal{M_k}$ for all $k$ up through subsets of size 7. 

**<font color=red>WARNING:</font>** This code might take a while, my computer took about 3 minutes to get through 7.  You can modify the code if you want to get up through 8, which took my computer a total of 6 minutes to get through but your results may vary. Now's a great time to hit enter and then take a stretch break.

In [ ]:
# Could take quite a while to complete... 

models_best = pd.DataFrame(columns=["RSS", "model"])

tic = time.time()
for i in range(1,8):
    models_best.loc[i] = getBest(i)

toc = time.time()
print("Total elapsed time:", (toc-tic), "seconds.")

In [ ]:
models_best

Notice that the `model` column actually is storing the model determined by stats model, so you can get it out to get info.

&#9989; **<font color=red>Do this:</font>** Without rerunning anything, what are the three variables used in $\mathcal{M}_3$? What is the $R^2$ value of the resulting model? What is the AIC of the resulting model?


In [ ]:
# Your code here #

The following script spits out the rsquared value for each model and returns it as a pandas series.

In [ ]:
# Gets the second element from each row ('model') and pulls out its rsquared attribute
models_best.apply(lambda row: row[1].rsquared, axis=1)

...and the next bit does this for all the test score approximations we talked about in class.....

In [ ]:
plt.figure(figsize=(20,10))
plt.rcParams.update({'font.size': 18, 'lines.markersize': 10})

# Set up a 2x2 grid so we can look at 4 plots at once
plt.subplot(2, 2, 1)

# We will now plot a red dot to indicate the model with the largest adjusted R^2 statistic.
# The argmax() function can be used to identify the location of the maximum point of a vector
plt.plot(models_best["RSS"])
plt.xlabel('# Predictors')
plt.ylabel('RSS')

# We will now plot a red dot to indicate the model with the largest adjusted R^2 statistic.
# The argmax() function can be used to identify the location of the maximum point of a vector

rsquared_adj = models_best.apply(lambda row: row[1].rsquared_adj, axis=1)

plt.subplot(2, 2, 2)
plt.plot(rsquared_adj)
plt.plot(rsquared_adj.argmax()+1, rsquared_adj.max(), "or")
plt.xlabel('# Predictors')
plt.ylabel('adjusted rsquared')

# We'll do the same for AIC and BIC, this time looking for the models with the SMALLEST statistic
aic = models_best.apply(lambda row: row[1].aic, axis=1)

plt.subplot(2, 2, 3)
plt.plot(aic)
plt.plot(aic.argmin()+1, aic.min(), "or")
plt.xlabel('# Predictors')
plt.ylabel('AIC')

bic = models_best.apply(lambda row: row[1].bic, axis=1)

plt.subplot(2, 2, 4)
plt.plot(bic)
plt.plot(bic.argmin()+1, bic.min(), "or")
plt.xlabel('# Predictors')
plt.ylabel('BIC')

# Forward selection

Ok, if nothing else, that amount of time that code takes to run is pretty frustrating.  We're going to implement forward selection as shown in Algorithm 6.2 in the book. First, let's figure out how to go through the for loop. Back to the fruits!

In [ ]:
my_set

Let's say that at step 2, I found out that the best model was the one that included `apples` and `bananas`. 

In [ ]:
M2 = ['apples','bananas']
remaining_predictors = [p for p in my_set if p not in M2]

print(M2)
print(remaining_predictors)

&#9989; **<font color=red>Do this:</font>** Print out all the size 3 sets of fruit-variables that will be checked at the next step in the forward algorithm. 


In [ ]:
# Your code here #

&#9989; **<font color=red>Do this:</font>** Modify this next function using what you figured out with the fruits. The input `predictors` is the list of variables `M_i`  from the hitters data set that were found to give the best model in the previous step.  


In [ ]:
def forward(predictors):

    # Pull out predictors we still need to process
    remaining_predictors = [p for p in X.columns if p not in predictors]
    
    tic = time.time()
    
    results = []
    
    #----------your code here----------#
    # For each of the new variable sets, use processSubset
    # to do the model, and add the output to the results 
    # data frame.
    #
    # Delete the line below here, it's just a placeholder so that the code 
    # runs before you add your new code 
    results.append({'model':None,"RSS":42}) # DELETE ME!!! #
    #-------------------------------#
    
    # Wrap everything up in a nice dataframe
    models = pd.DataFrame(results)
    
    # Choose the model with the highest RSS
    best_model = models.loc[models['RSS'].argmin()]
    
    toc = time.time()
    print("Processed ", models.shape[0], "models on", len(predictors)+1, "predictors in", (toc-tic), "seconds.")
    
    # Return the best model, along with some other useful information about the model
    return best_model

forward(['Hits','CRBI'])

And now we can wrap this party up in a huge for loop determine each $\mathcal{M}_i$

In [ ]:
models_fwd = pd.DataFrame(columns=["RSS", "model"])

tic = time.time()

# We start with no variables at all in our model
predictors = []

for i in range(1,len(X.columns)+1): 
    
    # Now we run our forward one step function from above to get 
    # the best model.     
    models_fwd.loc[i] = forward(predictors)
    
    # Then we extract the names of the variables that were used 
    # in that found model. When the for loop repeats, this will 
    # get sent back to predictors to the next stup.
    predictors = models_fwd.loc[i]["model"].model.exog_names
    
    # If you want to get something printed out to see which 
    # variables are used at each step, uncomment this line:
#     print('Starting with predictors:\n', predictors)


toc = time.time()
print("Total elapsed time:", round((toc-tic),2), "seconds.")

Look at that, no streatch break needed! On my computer that took about a half second. 

We can use the following code to figure out which variables were used for each $M_i$

In [ ]:
print(models_fwd.loc[1, "model"].model.exog_names)
print(models_fwd.loc[2, "model"].model.exog_names)

&#9989; **<font color=red>Q:</font>** Did the forward stepwise selection find the same model of size 6 as the earlier version that checked all subsets of size 6? How does the $R^2$ value compare between the two? (Note.... you hopefully haven't overwritten `models_best` so you shouldn't have to rerun any long for loops)


In [ ]:
# Your code here 

&#9989; **<font color=red>Q:</font>** Copy the figure plotting code from the best subsets model and modify it to see the various scores for your forward selection results. How does the size of the model chosen using each method ($R^2$, AIC, BIC) compare to the all subsets version?


In [ ]:
# Your code here 

# Backward Selection

Your final goal is to modify your forward function to get a new function `backward` which removes one variable at a time. 

&#9989; **<font color=red>Do this:</font>** Make the following function do what it should *a la* Algorithm 6.3.  That is, for a new list of predictors, write a for loop that removes one at a time, fits the model using `processSubset`, and adds the results to the data frame.

In [ ]:
# You modify this code

def backward(predictors):
    
    tic = time.time()
    
    results = []
    
    #----------your code here----------#
    # For each of the new variable sets, use processSubset
    # to fit the model, and add the output to the results 
    # data frame.
    #
    # Delete the line below here, it's just a placeholder so that the code 
    # runs before you add your new code 
    results.append({'model':None,"RSS":42}) # DELETE ME!!! #
    #-------------------------------#
    
    # Wrap everything up in a nice dataframe
    models = pd.DataFrame(results)
    
    # Choose the model with the highest RSS
    best_model = models.loc[models['RSS'].argmin()]
    
    toc = time.time()
    print("Processed ", models.shape[0], "models on", len(predictors)-1, "predictors in", (toc-tic), "seconds.")
    
    # Return the best model, along with some other useful information about the model
    return best_model

backward(['Hits','CRBI','CWalks','Errors'])

And again, we wrap this beast in a for loop and get a (potentially different) set of models $\mathcal{M}_k$ for each size $k$.

In [ ]:
models_bwd = pd.DataFrame(columns=["RSS", "model"], index = range(1,len(X.columns)))

tic = time.time()
predictors = X.columns

while(len(predictors) > 1):  
    models_bwd.loc[len(predictors)-1] = backward(predictors)
    predictors = models_bwd.loc[len(predictors)-1]["model"].model.exog_names

toc = time.time()
print("Total elapsed time:", (toc-tic), "seconds.")

&#9989; **<font color=red>Do this:</font>** Are all three models of size 7 (the one from all subsets, the one from the forward selection, and the one from backward selection) the same? How do the $R^2$ values compare?


In [ ]:
# Your code here #

&#9989; **<font color=red>Do this:</font>** Update the figure code above one more time to see which size set of variables is chosen by each test score method. If you have time, draw all three results on top of each other for easy visulalization.


In [ ]:
# Your code here #

# Lab Survey

To get credit for today's lab, fill out the following survey before the end of class:

https://forms.gle/hX8GT5FJ2fNMeTo1A

Note this is the same link for every lab, so you will fill this out multiple times this semester.



-----
### Congratulations, we're done!
Written by Dr. Liz Munch, Michigan State University

<a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-nc/4.0/88x31.png" /></a><br />This work is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by-nc/4.0/">Creative Commons Attribution-NonCommercial 4.0 International License</a>.